# Python Object-Oriented Programming

---

1. [Types of methods](#Methods-in-OOP),
    - [static method](#Static-method),
    - [class method](#Class-method),
2. [underscores in Python](#Underscore-functions-in-Python),
    - [last expression & skip expression](#Last-expression),
    - [private variables](#Private-variable-~weak-private),
    - [protected variables](#Protected-variable-~strong-private),
    - [keyword override](#Keyword-override),
    - [magic methods](#Magic-methods).
    
---

<img src="https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Ftse1.mm.bing.net%2Fth%3Fid%3DOIP.6MU4ScqH-1D93XRpyEYF4AHaHa%26pid%3DApi&f=1&ipt=03e1caf2482e1b90758340dcb676d181c012a20af163e9308f80f29ae4f6cc27&ipo=images" width="250" style="margin-left:auto; margin-right:auto"/>

## Methods in OOP

---
You can describe a **regular method** (also called an *instance method*) as a *user-defined function*..

In [202]:
class Employee:
    
    def __init__(self, first_name: str, last_name: str):
        self.first_name = first_name
        self.last_name = last_name
        
    def get_full_name(self) -> str:
        return f"{self.first_name} {self.last_name}"

<br>

...which belongs to the **class itself**.

So it is not possible to work with an *instance method* as with a standalone function:

In [203]:
get_full_name()  # BAD

NameError: name 'get_full_name' is not defined

In [204]:
matous = Employee("Matouš", "Holinka")

In [205]:
matous.get_full_name()

'Matouš Holinka'

<br>

A method that has, among its parameters, the special keyword `self`.

Thanks to that it can access both the class (and its *attributes*) and its *instances*:

In [206]:
print(matous.first_name, matous.last_name)

Matouš Holinka


<br>

However, the way of writing methods may differ.

There are several variants of using methods:
1. **instance** method,
2. **static** method,
3. **class** method.

<br>

### Static method

---

However, not every method necessarily has to belong to a class:

In [ ]:
class TextFileGenerator:
    def __init__(self, file_name: str):
        self.file_name = file_name
        self.file_content = list()

    def get_file_content(self, separator: str = '-', *content) -> str:
        return separator.join(content)

In such a case, it is unnecessary to use the `self` parameter, because it is not used anywhere in the `get_file_content` method.

Then it is good to ask yourself two simple questions:
1. **Is it necessary for it to be part of a class?**
2. **Where should I correctly place such a method?**

<br>

#### Placing the method outside the class

---

If it is an *object* that can be reused across **different classes** or **modules**, move the *method* outside the *class*.

It is more appropriate to turn it into a standalone *user-defined function*: 

In [207]:
def get_file_content(*content, separator: str = '-') -> str:
    return separator.join(content)


class TextFileGenerator:
    def __init__(self, file_name: str):
        self.file_name = file_name
        self.file_content = list()

In [208]:
txt_f = TextFileGenerator("muj_soubor.txt")

In [209]:
content = get_file_content("a", "b", "c")

In [210]:
print(
    txt_f.file_name,
    content,
    sep='\n'
)

muj_soubor.txt
a-b-c


<br>

If you need to change the separator, define it at the end (as a *keyword argument*).

In [211]:
content_2 = get_file_content("a", "b", "c", separator='#')

In [212]:
print(content_2)

a#b#c


<br>

#### Placing the method in the class with a decorator

---

If you want such a method to be **part of the class** (or it logically belongs there), you can keep it **inside the class**.

In that case, it is a good idea to tell others, **using a decorator**, that it is a *static method*.

In [213]:
from pathlib import Path

In [214]:
class TextFileGenerator:

    def __init__(self, file_name: str):
        self.file_content = list()
        self.file_name = file_name
            
    @staticmethod
    def exists_file(file_name: str) -> bool:
        return Path(file_name).is_file()

<br>

You can immediately recognize a *static method* by:
1. The **decorator** `@staticmethod`,
2. The **missing** `self` parameter.

It is a method that has a logical connection (is encapsulated) to a **specific class** or **instance** (easier to find, test, and document).

In [215]:
notes = TextFileGenerator("my_notes.txt")

In [217]:
notes.exists_file("my_notes.txt")

False

In this case your static method works perfectly.

There is **no** file `my_notes.txt` in the current directory!

In [218]:
# !ls -l  # Unix-based command
!dir

 Volume in drive C is OS
 Volume Serial Number is 0062-A493

 Directory of c:\Users\Radim Jedlicka\Documents\ENGETO\LEKTOR\Python_Akademie\Courses\python-5-3-2026-KBC\python-oop-kbc-en\notebooks\EN

03/10/2026  09:21 AM    <DIR>          .
03/08/2026  11:04 PM    <DIR>          ..
03/10/2026  09:04 AM            57,686 lesson01oop_en.ipynb
03/10/2026  09:21 AM            80,000 lesson02oop_en.ipynb
03/09/2026  09:36 PM            63,863 lesson03oop_en.ipynb
03/02/2026  09:42 PM            97,283 lesson04oop_en.ipynb
03/08/2026  11:03 PM               191 products.json
               5 File(s)        299,023 bytes
               2 Dir(s)  82,258,857,984 bytes free


<br>

But once you **create** it:

In [219]:
# !touch my_notes.txt # Unix-based command
!echo. > my_notes.txt

In [220]:
notes.exists_file("my_notes.txt")

True

Then the file will be found reliably.

You can also work without **creating an instance of the class**.

That is, just by using the class name itself where the *static method* is defined.

In [221]:
TextFileGenerator.exists_file("my_notes.txt")

True

<br>

**Attention**, this time you cannot work with just the method name!

In [222]:
exists_file("my_notes.txt")

NameError: name 'exists_file' is not defined

...because the *static method* itself **does not** have access **neither to instance attributes, nor to class attributes**:

In [223]:
class TextFileGenerator:

    def __init__(self, file_name: str):
        self.file_content = list()
        self.file_name = file_name
           
    @staticmethod
    def exists_file(file_name: str) -> bool:
        return Path(self.file_name).is_file()   # incorrectly placed parameter "self"

In [224]:
other_notes = TextFileGenerator("other_notes.txt")

In [225]:
other_notes.exists_file("other_notes.txt")

NameError: name 'self' is not defined

<br>

#### Recap of static methods

---

When to use the `@staticmethod` decorator:

1. When the method you created **will not be used in other classes**,
2. when it is **logically tied to the purpose of the class**.

In [226]:
class MathOperationSelector:
    """Object that groups various mathematical operations"""

    @staticmethod
    def add_two_numbers(x: int, y: int) -> int:
        return x + y
    
    @staticmethod
    def subtract_two_numbers(x: int, y: int) -> int:
        return x - y

In [227]:
addition = MathOperationSelector.add_two_numbers(4, 6)

In [228]:
subtraction = MathOperationSelector.subtract_two_numbers(121, 31)

In [229]:
print(addition, subtraction, sep='\n')

10
90


<br>

### 🧠 EXERCISE 🧠, Create a class `TextFileGenerator` that checks the file name suffix:
---

1. Copy the class `TextFileGenerator`,
2. keep only the instance attribute `get_file_name`,
3. write a **static method** `is_file_txt`,
4. check whether the parameter `file_name` has the extension `".txt"` or not,
5. if it has the `txt` suffix, return `True`, otherwise return `False`.

In [230]:
# Write your solution here
class TextFileGenerator:

    def __init__(self, file_name: str):
        self.content = list()
        self.file_name = file_name

    @staticmethod
    def is_file_txt(file_name: str, our_suffix: str = '.txt') -> bool:
        return Path(file_name).suffix == our_suffix


TextFileGenerator.is_file_txt("my_notes.txt")


True

In [231]:
TextFileGenerator.is_file_txt("my_file.txt")

True

In [232]:
TextFileGenerator.is_file_txt("my_file.csv")

False

<details>
    <summary>▶️ Solution</summary>
    
```python
class TextFileGenerator:

    def __init__(self, file_name: str):
        self.content = list()
        self.file_name = file_name

    @staticmethod
    def is_file_txt(file_name: str, suffix: str = '.txt') -> bool:
        return Path(file_name).suffix == suffix


TextFileGenerator.is_file_txt("my_notes.txt")
```
</details>

<br>

### Class method

---

Sometimes you need to adjust the **standard behaviour** of your class.

In [233]:
class Employee:
    def __init__(self, first_name: str, last_name: str, salary: int):
        self.first_name = first_name
        self.last_name = last_name
        self.salary = salary

<br>

Creating a new employee instance normally looks like this:

In [234]:
matous = Employee('Matouš', 'Holinka', 80_000)
karolina = Employee('Karolína', 'Šikovná', 100_000)

In [235]:
print(matous.salary, karolina.salary, sep='\n')

80000
100000


<br>

Sometimes it is not easy to connect to an existing system.

For example, if someone starts sending you data about new employees as a *table-like* or *text file*.

In [236]:
petr = "Petr;Svetr;110_000"
jan = "Jan;Novák;140_000"

<br>

How can I feed such inputs into the existing `Employee` class?

This is exactly a scenario for using **another decorator**, this time `@classmethod`.

In [237]:
class Employee:
    def __init__(self, first_name: str, last_name: str, salary: int):
        self.first_name = first_name
        self.last_name = last_name
        self.salary = salary
        
    @classmethod
    def from_separated_string(cls, input_string: str, separator: str = ';'):
        """
        Create a new instance of the `Employee` class from the given string.
        """
        try:
            first_name, last_name, salary = input_string.split(separator)
            
        except Exception:
            # logging.WARNING()
            instance = None
        else:
            instance = cls(first_name, last_name, salary)
        finally:
            return instance

<br>

A **class method** is easy to recognise:
1. The **decorator** `@classmethod`,
2. the **missing** helper parameter `self`,
3. a new helper parameter **`cls`** appears.

Instead of `cls` you can technically use any name.

But again, it is a convention so that other developers can easily see that this is a *class method*.

In [238]:
petr = "Petr;Svetr;110_000"
jan  = "Jan;Novák;140_000"

In [239]:
petr_i = Employee.from_separated_string(petr)

In [240]:
jan_i = Employee.from_separated_string(jan)

In [241]:
print(
    type(petr_i.salary),
    type(jan_i.salary),
    sep='\n'
)

<class 'str'>
<class 'str'>


The real benefit of the `@classmethod` decorator is that it lets you **create an alternative class constructor**.

<br>

Otherwise, you would have to **rewrite and split the values** manually, which is not very practical:

In [242]:
lukas = "Lukáš;Nový;90_000"

In [243]:
lukas = Employee(*lukas.split(";"))

In [244]:
print(
    lukas.first_name,
    lukas.last_name,
    lukas.salary,
    sep="\n"
)

Lukáš
Nový
90_000


In [245]:
class Employee:
    salary_increase = 1.06

    def __init__(self, first_name: str, last_name: str, salary: int):
        self.first_name = first_name
        self.last_name = last_name
        self.salary = salary

    @classmethod
    def increase_salary(cls, value: int):
        """
        Overwrite the value of the class attribute `salary_increase`.
        """
        cls.salary_increase = value

In [246]:
matous = Employee('Matouš', 'Holinka', 80_000)
karolina = Employee('Karolína', 'Šikovná', 100_000)

In [247]:
print(
    matous.salary_increase,
    karolina.salary_increase,
    sep="\n"
)

1.06
1.06


In [248]:
Employee.increase_salary(1.11)

In [249]:
print(
    matous.salary_increase,
    karolina.salary_increase,
    sep="\n"
)

1.11
1.11


You will most often see the `@classmethod` decorator on methods named something like:
- `from_`,
- `make_`,
- `create_`.

Their goal is simply to **make it easier to construct new instances** (from various files and inputs in general).

[Link, pandas library](https://github.com/pandas-dev/pandas/blob/55d78caa38925e8cf94623adcd0721cc15a56bdd/pandas/core/indexes/range.py#L171)

[Link, datetime module](https://github.com/python/cpython/blob/main/Lib/_pydatetime.py#L983)

### Summary of methods

---

1. **instance method** – can modify **both instance objects and class objects** (it sees both the class and the instance),
2. **static method (`@staticmethod`)** – cannot modify **either instance objects or class objects** (it sees neither class nor instance),
3. **class method (`@classmethod`)** – can modify class objects, but cannot modify instance objects (it sees the class, but not the instance).


<br>

### 🧠 EXERCISE 🧠, Create a class `Product` with the following requirements:
---

1. Define a class `Product` with the class attribute `count`,
2. in the instance constructor, pass the parameters `name`, `price` and `stock` in this order,
3. for every instance, increment `count` by 1,
4. create a method `add_to_stock` that takes a single parameter `count`,
5. create a method `sale` that checks whether the amount `count` can be sold,
6. create a static method `calculate_price_with_tax` that adds 21% tax to the price,
7. create a class method `from_json` that reads a JSON file and prepares an instance automatically.

In [ ]:
class Product:
    count = 0
    
    def __init__(self, name, price, stock):
        self.name = name
        self.price = price
        self.stock = stock
        Product.count += 1

    def sale(self, amount):
        if amount >= self.stock:
            return True
        else:
            return False
    
    def calculate_price_with_tax(self, tax = 1.21):
        return self.price * tax

    @classmethod
    def add_to_stock(cls, amount):
         cls.count += amount
    
    @classmethod
    def from_json(file_name):
        pass

In [ ]:
produkt1 = Product("Notebook", 20000, 10)
produkt2 = Product("Telefon", 15000, 15)
produkt3 = Product("Sluchátka", 2000, 50)

<details>
    <summary>▶️ Solution</summary>
    
```python
import json

class Product:
    count: int = 0

    def __init__(self, name: str, price: str, stock: int = 0):
        self.name = name
        self.price = price
        self.stock = stock
        Product.count += 1

    def add_to_stock(self, amount: int) -> None:
        self.stock += amount

    def sale(self, amount: int) -> None:
        if amount <= self.stock:
            self.stock -= amount
        else:
            print(f"There is not enough of product {self.name} in stock.")

    @classmethod
    def from_json(cls, file_name: str):
        try:
            with open(file_name, 'r') as f:
                json_content = json.load(f)
            
            # json_content je dict: {"0": {"name": "...", "price": ..., "amount": ...}}
            content = json_content["0"]  # nebo list(json_content.values())[0]
            
            instance = cls(
                content["name"],
                content["price"],
                content["amount"]
            )
        except Exception as e:
            print(f"Chyba: {e}")
            instance = None
        
        return instance

    @staticmethod
    def calculate_price_with_tax(price: float) -> float:
        """
        Return the product price including 21% tax.

        :param price: price without tax,
        :type price: float
        :return: price including tax,
        :rtype: float
        """
        return price * 1.21

```
</details>

In [ ]:
import json

class Product:
    count: int = 0

    def __init__(self, name: str, price: str, stock: int = 0):
        self.name = name
        self.price = price
        self.stock = stock
        Product.count += 1

    def add_to_stock(self, amount: int) -> None:
        self.stock += amount

    def sale(self, amount: int) -> None:
        if amount <= self.stock:
            self.stock -= amount
        else:
            print(f"There is not enough of product {self.name} in stock.")

    @classmethod
    def from_json(cls, file_name: str):
        try:
            with open(file_name, 'r') as f:
                json_content = json.load(f)
            
            # json_content je dict: {"0": {"name": "...", "price": ..., "amount": ...}}
            content = json_content["0"]  # nebo list(json_content.values())[0]
            
            instance = cls(
                content["name"],
                content["price"],
                content["amount"]
            )
        except Exception as e:
            print(f"Chyba: {e}")
            instance = None
        
        return instance

    @staticmethod
    def calculate_price_with_tax(price: float) -> float:
        """
        Return the product price including 21% tax.

        :param price: price without tax,
        :type price: float
        :return: price including tax,
        :rtype: float
        """
        return price * 1.21


In [ ]:
apple = Product("Apple", 10.0, 100)
beer = Product("Beer", 20.0, 50)

In [ ]:
Product.count

In [ ]:
apple.add_to_stock(50)

In [ ]:
apple.stock

In [ ]:
apple.sale(100)

In [ ]:
beer.calculate_price_wit_tax(100)

In [ ]:
Product.calculate_price_with_tax(100)

In [ ]:
product = Product.from_json("products.json")
print(product.name, product.price, product.stock)

<img src="https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Ftse1.mm.bing.net%2Fth%3Fid%3DOIP.W1bO-UE-mgx7us8eNrJTsQHaHa%26pid%3DApi&f=1&ipt=eaddf3d0ad49228dbd8d3a969c2c025036cee17f5c70daad917debf57a2f743e&ipo=images" width="250" style="margin-left:auto; margin-right:auto"/>

## Underscore functions in Python

---

The underscore is a **syntactic character** in Python that has several important meanings:
1. last expression `_`,
2. ignore an expression `_`,
3. private variable `_name`,
4. protected variable `__name`,
5. overridden keyword `class_`,
6. magic method `__init__`.

<br>

### Last expression

---

The first (although not very practical) use is to access the value of the last expression:

In [1]:
"Marek"

'Marek'

In [2]:
_

'Marek'

In [3]:
_.upper()

'MAREK'

In [4]:
1 + 119

120

In [5]:
_

120

In [6]:
_ + 30

150

<br>

You will appreciate this behaviour mainly in *notebook* and *interpreter* environments.

In a source file, it is almost never used this way.


### Ignore an expression

---

The underscore becomes really useful in scripts as an **indicator of unused values**:

In [7]:
import time

In [8]:
for item in range(5):  # BAD
    print("Checking status..")
    time.sleep(1)

Checking status..
Checking status..
Checking status..
Checking status..
Checking status..


In [9]:
for _ in range(5):     # EXCELLENT
    print("Checking status..")
    time.sleep(1)

Checking status..
Checking status..
Checking status..
Checking status..
Checking status..


<br>

Giving a name to the loop variable in the example above makes no sense.

You **do not need to use it** anywhere in the loop body.

It is therefore a good idea to signal this to other readers of the code.

<br>

Another use case is *iterating* over multiple values when you only need **one specific value**:

In [10]:
for index, first_name in enumerate(("Matouš", "Marcel", "Filip")):  # BAD
    print(index)

0
1
2


In [11]:
for index, _ in enumerate(("Matouš", "Marcel", "Filip")):      # EXCELLENT
    print(index)

0
1
2


<br>

Again, other developers can now immediately see that the essential work in the loop body is done with the index.

<br>

Another common pattern is **multiple assignment**:

In [12]:
first_name, last_name, salary = "Filip;Svědomitý;90_000".split(";")  # BAD

<br>

Again, there is nothing fundamentally wrong with this code.

But if you only need to unpack the value into the variable `jmeno`:

In [13]:
first_name, _, _ = "Filip;Svědomitý;90_000".split(";")            # GOOD

In [14]:
print(first_name)

Filip


<br>

Alternatively, you can pack the remaining unused values into a `list`:

In [15]:
first_name, *_ = "Filip;Svědomitý;90_000".split(";")              # EXCELLENT

In [16]:
print(first_name)

Filip


In [17]:
print(_)

['Svědomitý', '90_000']


<br>

### Protected variable (~weak private)

---

You may know the terms **protected** and **public variable** from other languages.

Some languages support truly **internal** (*private*) variables (e.g. **Java**, **C#**):

In [18]:
class LogValidator:
    def __init__(self, file_name: str):
        self.file_name = file_name
        self.active = True

    def get_status(self):
        if self.active:
            print(f"Checking log file: {self.file_name}")

In [19]:
validator_1 = LogValidator("file.log")

In [20]:
validator_1.get_status()

Checking log file: file.log


<br>

If you add a single underscore as a *prefix* to the object name, you mark it as a *protected object*.

Python does not enforce protected variables like some other languages.

It treats this convention only **as a hint** or an indicator.

In [21]:
class LogValidator:
    def __init__(self, file_name: str):
        self.file_name = file_name
        self._active = True  # WEAK PRIVATE

    def get_status(self):
        if self._active:
            print(f"Checking log file: {self.file_name}")
        else:
            raise Exception("Uh-oh, some unexpected behaviour")

In [22]:
validator_2 = LogValidator("file.log")

In [23]:
validator_2.get_status()

Checking log file: file.log


In [24]:
print(validator_2._active)

True


In [25]:
print(validator_2.active)

AttributeError: 'LogValidator' object has no attribute 'active'

<br>

The single underscore here works **only as an indicator**.

It tells other readers of the source code that this is a **protected variable**.

In other words, it is an **internal object** – *DO NOT MODIFY* it.

You may use it, but only within **one class (and its subclasses) or one module**.

<br>

It is usually a good idea to document what such an attribute is used for and how to work with it from the outside.

However, Python is very permissive and the *interpreter* will allow you to **use and overwrite** the variable anyway.

In [26]:
validator_2._active = False  # BAD

In [27]:
validator_2.get_status()

Exception: Uh-oh, some unexpected behaviour

<br>

If you overwrite such a value, you may change the behaviour of the script, which is not desirable.

In the example above, a shorter time for checking the log might not be sufficient.

<br>

### Private variable (~strong private)

---

Using **two underscores** you can define **private** variables.

Such a variable (object) should be used **only within the class**:

In [28]:
from time import sleep

In [29]:
class LogValidator:
    """Object representing a log file."""
    
    def __init__(self, file_name: str):
        self.file_name = file_name
        self._limit = 3                                # WEAK PRIVATE
        self.notification = self.__set_notification()  # STRONG PRIVATE
        
    def check_log(self) -> None:
        print(self.notification)

        for _ in range(self._limit):
            print(f"Checking log file..")
            sleep(1)
        else:
            print(f"File: '{self.file_name}' is OK.")
            
    def __set_notification(self):
        return "Default notification"

<br>

This setup further highlights the **importance of the variable**.

It makes it harder for the user to simply overwrite or re-type the expression (to access it directly):

In [30]:
validator_3 = LogValidator("file.log")

In [31]:
print(validator_3._limit)

3


In [32]:
validator_3.check_log()

Default notification
Checking log file..
Checking log file..
Checking log file..
File: 'file.log' is OK.


<br>

This way the user will not find it so easily.

In the background, Python performs so-called *name mangling* (something like **name scrambling**).

It is still possible to look up the attribute manually and put in the effort to overwrite it:

In [33]:
# Practical Demonstration of Name Mangling

class LogValidator:
    def __init__(self):
        self.__limit = 3  # STRONG PRIVATE - will be renamed!
        
validator = LogValidator()

# ❌ This DOES NOT work - Python can't find __limit:
try:
    print(validator.__limit)
except AttributeError as e:
    print(f"Error: {e}")

print("\n--- What Python does internally ---")
print(f"Python renames __limit to: _LogValidator__limit")

# ✅ But THIS works - using the internal name that Python created:
print(f"validator._LogValidator__limit = {validator._LogValidator__limit}")

# You CAN also change it if you know the internal name:
validator._LogValidator__limit = 10
print(f"After change: validator._LogValidator__limit = {validator._LogValidator__limit}")

Error: 'LogValidator' object has no attribute '__limit'

--- What Python does internally ---
Python renames __limit to: _LogValidator__limit
validator._LogValidator__limit = 3
After change: validator._LogValidator__limit = 10


In [ ]:
print(validator_3.notification)

<br>

The magic attribute `__dict__` allows you to list all instance attributes, where you will also find protected variables:

In [34]:
class LogValidator:
    """Object representing a log file."""
    
    def __init__(self, file_name: str):
        self.file_name = file_name
        self.__limit = 3  # STRONG PRIVATE
        
    def check_log(self) -> None:
        for _ in range(self.__limit):
            print(f"Checking log file..")
            sleep(1)
        else:
            print(f"File: '{self.file_name}' is OK.")

In [35]:
validator_4 = LogValidator("file.log")

In [36]:
print(validator_4.__limit)

AttributeError: 'LogValidator' object has no attribute '__limit'

In [37]:
print(validator_4.__dict__)

{'file_name': 'file.log', '_LogValidator__limit': 3}


In [38]:
validator_4._LogValidator__limit = 10

In [39]:
print(validator_4._LogValidator__limit)

10


<br>

The *interpreter* protects such an attribute by **implicitly renaming** it.

#### 🧠 EXERCISE 🧠, Create a class `Employee` that contains:

---

1. The constructor `__init__` sets only the instance attribute `name`,
2. `__init__` also creates a *weak private* attribute `age` and a *strong private* attribute `salary`, both set to `None`,
3. write a method `set_age` that takes the parameter `age: int`,
4. the method `set_age` sets the *weak private* attribute `age` if the argument is `18 < x < 65` (otherwise it raises `Exception`),
5. write a method `set_salary` that takes the parameter `salary: int`,
6. the method `set_salary` sets the *strong private* attribute `salary` if the argument is `x > 34_000` (otherwise it raises `Exception`),
7. create an instance method `show_info` that prints: `"Name: <NAME>, Age: <AGE>"`,
8. create a *weak private* instance method `_calculate_tax` that calculates 20% of the salary.

In [40]:
class Employee:
    
    def __init__(self, name):
        self.name = name
        self._age = None
        self.__salary = None
    
    def set_age(self, age: int):
        if 18 < age < 65:
            self._age = age
        else:
            raise Exception("Age does not meet the requirements!")
    
    def set_salary(self, salary: int):
        if salary > 34_000:
            self.__salary = salary
        else: 
            raise Exception("Salary not set because it is lower than 34 000!")
        
    def show_info(self):
        print(f"{self.name}, {self._age} years old")
    
    def _calculate_tax(self, tax = 0.2):
        return self.__salary * tax

In [41]:
matous = Employee("Matouš")

In [42]:
print(matous._age)

None


In [43]:
matous.set_age(50)

In [44]:
print(matous._age)

50


In [45]:
matous.set_salary(50_000)

In [46]:
matous.set_salary(20_000)

Exception: Salary not set because it is lower than 34 000!

In [47]:
matous.show_info()

Matouš, 50 years old


In [48]:
matous._calculate_tax()

10000.0

<br>

### Overriding a keyword

---

If a **keyword** collides with a variable name, the *interpreter* will raise a **syntax error**:

In [49]:
class Employee:

    def __init__(self, first_name: str, email: str):
        self.first_name = first_name
        self.email = email

In [50]:
class = Employee("Matous", "matous@gmail.com")  # reserved expression 'class'

SyntaxError: invalid syntax (2334282273.py, line 1)

<br>

To avoid this, you can add an underscore as a suffix to the keyword:

In [51]:
class_ = Employee("Matous", "matous@gmail.com")  # str_, print_

In [52]:
print(
    class_.first_name,
    class_.email,
    sep="\n"
)

Matous
matous@gmail.com


<br>

### Magic methods

---

Magic methods are **special methods** in Python (~*double-underscore methods* = **dunder** methods).

Among magic methods is also `__init__`, the *instance constructor*.

<br>

These methods are like the gears that drive Python under the hood:

In [53]:
len((0.1, 0.2, 0.3))

3

In [54]:
(0.1, 0.2, 0.3).__len__()  # len()

3

In [55]:
number = 3

In [56]:
number.__eq__(3)  # ==

True

In [57]:
number.__eq__(4)  # ==

False

In [58]:
number.__add__(7) # +

10

<br>

The functionality of many objects depends on these methods behind the scenes.

In [59]:
class Employee:

    def __init__(self, first_name: str, email: str):
        self.first_name = first_name
        self.email = email

In [60]:
matous = Employee("Matous", "matous@gmail.com")

In [61]:
print(
    matous,
    str(matous),
    repr(matous),
    sep="\n"
)

<br>

If you need to customise the **informational output** for an instance of the `Employee` class, you can override the `__str__` method:

In [62]:
class Employee:

    def __init__(self, first_name: str, email: str):
        self.first_name = first_name
        self.email = email
        
    def __str__(self) -> str:
        return f"Employee name: {self.first_name}"
    
    # def __repr__(self):
    #     pass

In [63]:
matous = Employee("Matous", "matous@gmail.com")

In [64]:
print(
    matous,
    str(matous),
    repr(matous),
    sep="\n"
)

Employee name: Matous
Employee name: Matous


<br>

You can represent an object in two main ways:
1. overriding the `__str__` method (informal, mainly for end users),
2. overriding the `__repr__` method (formal, for logs and debugging).

In [ ]:
import datetime

In [ ]:
current_date_time = datetime.datetime.now()

In [ ]:
current_date_time        # __repr__

In [ ]:
repr(current_date_time)        # __repr__

In [ ]:
print(current_date_time) # __str__

In [ ]:
str(current_date_time)   # __str__

In [ ]:
type(current_date_time)

<br>

If you want to apply your own methods in your class:

In [ ]:
class Employee:

    def __init__(self, first_name: str, email: str):
        self.first_name = first_name
        self.email = email
        
    def __str__(self) -> str:
        return f"Employee name: {self.first_name}"
    
    def __repr__(self):
        return f"{type(self).__name__}(name={self.first_name!r})"

In [ ]:
matous = Employee("Matous", "matous@gmail.com")

In [ ]:
print(
    str(matous),
    repr(matous),
    sep="\n"
)

In [ ]:
matous.__dict__

<br>

Using **magic methods** is one of the more advanced features of OOP.

#### Overloading of magic methods

---

Adding integers and floats is well known:

In [ ]:
print(11 + 9)

<br>

Similarly, you can imagine *concatenating* strings:

In [ ]:
print('Matouš' + ' Holinka')

<br>

But what if you need to **add two-dimensional vectors**:

In [ ]:
class Vector2D:
    def __init__(self, dimension_x: int, dimension_y: int):
        self.dimension_x = dimension_x
        self.dimension_y = dimension_y

In [ ]:
v1 = Vector2D(dimension_x=5, dimension_y=4)
v2 = Vector2D(dimension_x=7, dimension_y=6)

In [ ]:
print(v1 + v2)

<br>

In this case, addition is not possible.

For the `Vector2D` object, addition is not supported yet.

In [ ]:
class Vector2D:
    def __init__(self, dimension_x: int, dimension_y: int):
        self.dimension_x = dimension_x
        self.dimension_y = dimension_y
        
    def __add__(self, other: Vector2D):  # +
        return [
            self.dimension_x + other.dimension_x,  # Vector2D(x) + Vector2D(x) = 5 + 7
            self.dimension_y + other.dimension_y   # Vector2D(y) + Vector2D(y) = 4 + 6
        ]

In [ ]:
v1 = Vector2D(dimension_x=5, dimension_y=4)
v2 = Vector2D(dimension_x=7, dimension_y=6)

In [ ]:
print(v1 + v2)

<br>

### 🧠 EXERCISE 🧠, Create a class `ShoppingCart` with the following requirements:
---

1. Define a class `ShoppingCart`,
2. prepare an instance constructor that **has no parameters**,
3. update the constructor to initialize a *weak private* attribute `items` as a `dict`,
4. define a magic method `__str__` that returns `"Shopping cart: <SHOPPING_CART_ITEMS>"`,
5. create an instance method `add_item` with parameters `name: str` and `quantity: int`,
6. in `add_item`, first validate the `name` parameter using another static method `validate_item`,
7. if `validate_item` confirms that `name` is a non-empty `str`, return `True`; otherwise return `False`,
8. if the static method `validate_item` returns `True`, then `add_item` stores the value in `items` as `items[name] = quantity`,
9. if the product already exists in `items`, **increment** the existing value by the new value,
10. create an instance method `remove_item` with parameter `name: str`,
11. if `name` is present in `items`, remove it,
12. create a class method `create_with_sample_items`,
13. `create_with_sample_items` should add **5 apples** and **3 pears** to the original empty `dict` named `items`.

In [ ]:
cart = ShoppingCart.create_with_sample_items()

In [ ]:
print(cart)

In [ ]:
cart.add_item("bananas", 2)

In [ ]:
print(cart)

In [ ]:
cart.remove_item("pears")
print(cart)

In [ ]:
print(cart)

<details>
    <summary>▶️ Solution</summary>
    
```python
class ShoppingCart:
    def __init__(self):
        self._items = {}

    def __str__(self):
        return f"My shopping cart: {self._items}"

    def add_item(self, name: str, count: int) -> None:
        if self.validate_item(name):
            self._items[name] = self._items.get(name, 0) + count

    def remote_item(self, name: str):
        if name in self._items:
            del self._items[name]

    @staticmethod
    def validate_item(name: str):
        return isinstance(name, str) and name != ""

    @classmethod
    def create_with_sample_items(cls):
        cart = cls()
        cart.add_item("apples", 5)
        cart.add_item("pears", 3)
        return kosik
```
</details>

<br>

### 🧠 EXERCISE 🧠, Create a class `User` with the following requirements:
---

- Define a class `User`,
- prepare an instance constructor that takes parameters `owner` and `last_active`,
- make the instance attribute `owner` protected,
- create a method `show_user` that returns the user name,
- create a method `user_login` that creates a **private instance attribute** `current_active` to store the current date and time,
- create a method `show_login` that prints the user's current login time (unformatted),
- create the magic method `__str__` that formats the `str` as: `User: <owner> is active since: <dd/mm/YYYY HH:MM:SS>.`,
- create a static method `is_active` that takes parameters `last_login` and `current_login`,
- the method `is_active` returns `True`/`False` if the difference between `_current_active` and `last_active` is less than 365 days.

In [ ]:
from pandas import to_datetime, Timestamp

<details>
    <summary>▶️ Solution</summary>
    
```python
class User:
    """
    Object representing a user account for a
    fictional web project for buying books.
    """

    def __init__(self, owner: str, last_active):
        self._owner = owner
        self.last_active = Timestamp(last_active)
        self.__current_active = None

    def user_login(self):
        self.__current_active = to_datetime('today')

    def show_login(self):
        return self.__current_active

    def show_user(self):
        return self._owner

    def __str__(self) -> str:
        active_since = self.show_login() if self.show_login() is not None else self.last_active
        return f"User: {self.show_user()} is active since: {active_since.strftime('%d/%m/%Y %H:%M:%S')}."

    @staticmethod
    def is_active(last_login, current_login) -> bool:
        """
        Return True if the user last logged in less than 365 days ago,
        otherwise return False.
        """
        return (Timestamp(current_login) - Timestamp(last_login)).days < 365
```
</details>

---